# Whisper STT Implementation

## Step 1: Setting Up the Environment

In [1]:
!uv pip install openai pydub python-dotenv

Using Python 3.12.14 environment at: /Users/ben-macbook-air/Files/Projects/Ironhack Bootcamp/.venv
Checked 3 packages in 31ms


In [2]:
import os
import shutil
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# API client (key comes from .env, never hard-code it)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Project directories
AUDIO_DIR = Path("recordings")
CHUNKS_DIR = Path("chunks")
OUTPUT_DIR = Path("output")
for d in (AUDIO_DIR, CHUNKS_DIR, OUTPUT_DIR):
    d.mkdir(exist_ok=True)

# Checkpoint: key loaded, ffmpeg available, directories exist
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("ffmpeg found:", shutil.which("ffmpeg"))
print("Directories:", [str(d) for d in (AUDIO_DIR, CHUNKS_DIR, OUTPUT_DIR)])
print("Audio files:", [f.name for f in AUDIO_DIR.glob("*")])

API key loaded: True
ffmpeg found: /opt/homebrew/bin/ffmpeg
Directories: ['recordings', 'chunks', 'output']
Audio files: ['project_nimbus_meeting.mp3', 'meeting_sample.wav']


## Step 2: Sample Meeting Audio

In [3]:
from pydub import AudioSegment
from IPython.display import Audio, display

AUDIO_PATH = AUDIO_DIR / "project_nimbus_meeting.mp3"
RECORDING_NAME = AUDIO_PATH.stem   # used for chunk folder and output file names

# Checkpoint: file exists and can be loaded
assert AUDIO_PATH.exists(), f"Audio file not found: {AUDIO_PATH}"
meeting_audio = AudioSegment.from_file(AUDIO_PATH)

size_mb = AUDIO_PATH.stat().st_size / (1024 * 1024)
print(f"File:        {AUDIO_PATH.name}")
print(f"Size:        {size_mb:.2f} MB (Whisper API limit: 25 MB)")
print(f"Duration:    {len(meeting_audio) / 1000:.1f} s ({len(meeting_audio) / 60000:.1f} min)")
print(f"Channels:    {meeting_audio.channels}")
print(f"Sample rate: {meeting_audio.frame_rate} Hz")

# Listen to the first 10 seconds
display(Audio(meeting_audio[:10_000].export(format="wav").read(), rate=meeting_audio.frame_rate))

File:        project_nimbus_meeting.mp3
Size:        2.36 MB (Whisper API limit: 25 MB)
Duration:    309.2 s (5.2 min)
Channels:    1
Sample rate: 22050 Hz


## Step 3: Basic Transcription (Without Chunking)

In [4]:
import io

# Take the first 60 seconds of the meeting
short_clip = meeting_audio[:60_000]

# Export to an in-memory mp3 (Whisper needs a file-like object with a name)
clip_buffer = io.BytesIO()
short_clip.export(clip_buffer, format="mp3")
clip_buffer.seek(0)
clip_buffer.name = "clip_60s.mp3"

print("Transcribing first 60 seconds...")
basic_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=clip_buffer,
)

# Checkpoint: inspect the response object and the text
print("Response type:", type(basic_transcript).__name__)
print("Fields:", list(basic_transcript.model_dump().keys()))
print("\nTranscription:")
print("-" * 40)
print(basic_transcript.text)

Transcribing first 60 seconds...
Response type: Transcription
Fields: ['text', 'languages', 'logprobs', 'usage']

Transcription:
----------------------------------------
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number nimpash482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team?


## Step 4: Transcription with Prompts (Guided Approach)

In [5]:
def to_buffer(segment, name="clip.mp3"):
    """Export a pydub AudioSegment to an in-memory mp3 that the API accepts."""
    buf = io.BytesIO()
    segment.export(buf, format="mp3")
    buf.seek(0)
    buf.name = name
    return buf


# Context prompt: names and jargon that appeared (or were mangled) in the plain transcript
MEETING_PROMPT = (
    "Weekly engineering sync for Project Nimbus. Participants: Marcus, Priya. "
    "Topics: Kubernetes migration, EKS cluster, Helm, Redis cache, "
    "persistent volume claims, storage class, PostgreSQL latency, Q3 OKRs, "
    "DevOps team, Jira ticket."
)

print("Transcribing first 60 seconds WITH prompt...")
prompted_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=to_buffer(short_clip, "clip_60s.mp3"),
    prompt=MEETING_PROMPT,
)

print("\nTranscription (prompted):")
print("-" * 40)
print(prompted_transcript.text)

# Save so we can compare later
(OUTPUT_DIR / f"{RECORDING_NAME}_clip_prompted.txt").write_text(prompted_transcript.text)

Transcribing first 60 seconds WITH prompt...

Transcription (prompted):
----------------------------------------
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the Redis cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number NIM-482, and I expect it to be fixed by Tuesday. OK, good. Do we need anything from the DevOps team?


733

## Step 5: Transcription Without Prompts (Unguided Approach)

In [6]:
import difflib

# Unprompted run: same call as Step 3, re-run here so both versions are saved side by side
print("Transcribing first 60 seconds WITHOUT prompt...")
unprompted_transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=to_buffer(short_clip, "clip_60s.mp3"),
)
(OUTPUT_DIR / f"{RECORDING_NAME}_clip_unprompted.txt").write_text(unprompted_transcript.text)

prompted_text = prompted_transcript.text
unprompted_text = unprompted_transcript.text

print("\nUNPROMPTED:\n" + unprompted_text)
print("\nPROMPTED:\n" + prompted_text)

# Word-level diff: only show where the two versions disagree
a, b = unprompted_text.split(), prompted_text.split()
matcher = difflib.SequenceMatcher(None, a, b)

print("\nDifferences (unprompted -> prompted):")
print("-" * 40)
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag != "equal":
        print(f"{' '.join(a[i1:i2])!r:30} -> {' '.join(b[j1:j2])!r}")

print(f"\nSimilarity: {matcher.ratio():.1%}")

Transcribing first 60 seconds WITHOUT prompt...

UNPROMPTED:
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the authentication service, and the billing service to the new EKS cluster last week. The main blocker right now is the RIDIS cache. When we deployed it with Helm, the persistent volume claims were failing, because of a misconfigured storage class. I opened a ticket, Jira number nimpash482, and I expect it to be fixed by Tuesday. Okay, good. Do we need anything from the DevOps team?

PROMPTED:
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL latency issu

## Step 6: Implementing Audio Chunking

In [7]:
def split_audio(audio, out_dir, chunk_length_minutes=1.0, prefix="chunk"):
    """Split an AudioSegment into fixed-length chunks saved as mp3 files.

    Returns a list of dicts with the file path and the chunk's start offset in
    seconds, which Step 7 needs to correct per-chunk timestamps.
    """
    chunk_ms = int(chunk_length_minutes * 60 * 1000)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Remove chunks from earlier runs so stale files don't get mixed in
    for old in out_dir.glob(f"{prefix}_*.mp3"):
        old.unlink()

    chunks = []
    for i, start_ms in enumerate(range(0, len(audio), chunk_ms)):
        piece = audio[start_ms:start_ms + chunk_ms]
        path = out_dir / f"{prefix}_{i:03d}.mp3"
        piece.export(path, format="mp3")
        chunks.append({
            "index": i,
            "path": path,
            "offset": start_ms / 1000,        # seconds into the full recording
            "duration": len(piece) / 1000,
        })
    return chunks


# Using 1-minute chunks so our 5-minute file produces several (real meetings: 10+ min)
chunk_dir = CHUNKS_DIR / RECORDING_NAME   # one subfolder per recording
chunk_info = split_audio(meeting_audio, chunk_dir, chunk_length_minutes=1)

# Checkpoint: chunks exist, sizes are sensible, durations add up to the original
print(f"Created {len(chunk_info)} chunks in '{chunk_dir}/'\n")
for c in chunk_info:
    size_kb = c["path"].stat().st_size / 1024
    print(f"{c['path'].name}  offset={c['offset']:6.1f}s  duration={c['duration']:5.1f}s  size={size_kb:6.1f} KB")

total = sum(c["duration"] for c in chunk_info)
print(f"\nTotal chunk duration: {total:.1f}s vs original {len(meeting_audio) / 1000:.1f}s")

Created 6 chunks in 'chunks/project_nimbus_meeting/'

chunk_000.mp3  offset=   0.0s  duration= 60.0s  size= 234.8 KB
chunk_001.mp3  offset=  60.0s  duration= 60.0s  size= 234.8 KB
chunk_002.mp3  offset= 120.0s  duration= 60.0s  size= 234.8 KB
chunk_003.mp3  offset= 180.0s  duration= 60.0s  size= 234.8 KB
chunk_004.mp3  offset= 240.0s  duration= 60.0s  size= 234.8 KB
chunk_005.mp3  offset= 300.0s  duration=  9.2s  size=  36.4 KB

Total chunk duration: 309.2s vs original 309.2s


## Step 7: Transcribing Chunks with Timestamps

In [8]:
import time


def fmt_time(seconds):
    """Format seconds as HH:MM:SS."""
    s = int(seconds)
    return f"{s // 3600:02d}:{(s % 3600) // 60:02d}:{s % 60:02d}"


def transcribe_chunks(chunks, prompt=None, delay=0.5, retries=3):
    """Transcribe each chunk with timestamps and shift them to full-recording time."""
    segments, words = [], []

    for c in chunks:
        print(f"Transcribing {c['path'].name} ({c['index'] + 1}/{len(chunks)})...")

        for attempt in range(retries):
            try:
                with open(c["path"], "rb") as f:
                    result = client.audio.transcriptions.create(
                        model="whisper-1",
                        file=f,
                        response_format="verbose_json",
                        timestamp_granularities=["segment", "word"],
                        **({"prompt": prompt} if prompt else {}),
                    )
                break
            except Exception as e:
                if attempt == retries - 1:
                    raise
                wait = 2 ** (attempt + 1)
                print(f"  error ({e}); retrying in {wait}s")
                time.sleep(wait)

        # Timestamps are relative to the chunk, so add the chunk's offset
        for seg in result.segments:
            segments.append({
                "chunk": c["index"],
                "start": seg.start + c["offset"],
                "end": seg.end + c["offset"],
                "text": seg.text.strip(),
            })
        for w in result.words:
            words.append({
                "chunk": c["index"],
                "word": w.word,
                "start": w.start + c["offset"],
                "end": w.end + c["offset"],
            })

        time.sleep(delay)  # gentle on rate limits

    return segments, words


all_segments, all_words = transcribe_chunks(chunk_info, prompt=MEETING_PROMPT)

# Checkpoint: combined transcript with full-recording timestamps
print(f"\n{len(all_segments)} segments, {len(all_words)} words\n")
for seg in all_segments:
    print(f"[{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['text']}")

# Sanity check: timestamps should be increasing and stay within the recording length
starts = [s["start"] for s in all_segments]
print("\nMonotonic starts:", starts == sorted(starts))
print(f"Last segment ends at {all_segments[-1]['end']:.1f}s (recording is {len(meeting_audio) / 1000:.1f}s)")

Transcribing chunk_000.mp3 (1/6)...
Transcribing chunk_001.mp3 (2/6)...
Transcribing chunk_002.mp3 (3/6)...
Transcribing chunk_003.mp3 (4/6)...
Transcribing chunk_004.mp3 (5/6)...
Transcribing chunk_005.mp3 (6/6)...

51 segments, 635 words

[00:00:00 - 00:00:06] Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today
[00:00:06 - 00:00:13] is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL
[00:00:13 - 00:00:20] latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the
[00:00:21 - 00:00:21] 
[00:00:21 - 00:00:29] migration? Sure, thanks Marcus. So the Kubernetes migration is about 70% done. We moved the
[00:00:29 - 00:00:36] authentication service, and the billing service to the new EKS cluster last week. The main
[00:00:36 - 00:00:42] blocker right now is the Redis cache. When we deployed it with Helm, the persistent volume
[00:00:42 - 00:00:48] claims were failing, because of a misconfig

## Step 8: Exporting with Timestamps

In [9]:
import json


def non_empty(segments):
    """Whisper occasionally returns empty segments (e.g. at pauses); drop them for readable exports."""
    return [s for s in segments if s["text"].strip()]


def export_txt(segments, path):
    lines = [f"[{fmt_time(s['start'])} - {fmt_time(s['end'])}] {s['text']}" for s in non_empty(segments)]
    Path(path).write_text("\n".join(lines) + "\n", encoding="utf-8")


def export_json(segments, words, path, source, prompt):
    segments = non_empty(segments)
    data = {
        "source": str(source),
        "model": "whisper-1",
        "prompt": prompt,
        "duration": segments[-1]["end"] if segments else 0,
        "full_text": " ".join(s["text"] for s in segments),
        "segments": segments,
        "words": words,
    }
    Path(path).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


export_txt(all_segments, OUTPUT_DIR / f"{RECORDING_NAME}_transcript.txt")
export_json(all_segments, all_words, OUTPUT_DIR / f"{RECORDING_NAME}_transcript.json", AUDIO_PATH, MEETING_PROMPT)

# Checkpoint: files exist and are readable
for ext in ("txt", "json"):
    p = OUTPUT_DIR / f"{RECORDING_NAME}_transcript.{ext}"
    print(f"{p}  ({p.stat().st_size / 1024:.1f} KB)")

print("\nTXT preview:")
print("".join(open(OUTPUT_DIR / f"{RECORDING_NAME}_transcript.txt").readlines()[:5]))

# Simple search over the exported JSON: where was a term mentioned?
data = json.loads((OUTPUT_DIR / f"{RECORDING_NAME}_transcript.json").read_text(encoding="utf-8"))
term = "terraform"
print(f"Mentions of '{term}':")
for s in data["segments"]:
    if term in s["text"].lower():
        print(f"  [{fmt_time(s['start'])}] {s['text']}")

output/project_nimbus_meeting_transcript.txt  (4.6 KB)
output/project_nimbus_meeting_transcript.srt  (5.2 KB)
output/project_nimbus_meeting_transcript.json  (86.3 KB)

SRT preview:
1
00:00:00,000 --> 00:00:06,080
Good morning everyone. This is the weekly engineering sync for Project Nimbus. Today

2
00:00:06,080 --> 00:00:13,060
is Thursday, and we have four items on the agenda. The Kubernetes migration, the PostgreSQL

3
00:00:13,060 --> 00:00:20,280
latency issue, the Q3 OKRs, and the hiring update. Priya, do you want to start with the


Mentions of 'terraform':
  [00:01:00] Yes, we need Thomas to approve the Terraform changes for the VPC peering. Without that,
  [00:01:12] I can review the Terraform pull request this afternoon. I saw it in GitHub yesterday, but
  [00:04:48] Thomas will review the Terraform pull request today, and deploy the index migration.
